# Notebook 03 — Model Experiments & Comparison

Train all 3 models, compare metrics, plot confusion matrices.

In [ ]:
import sys; sys.path.insert(0,'..')
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from src.config import PROCESSED_DATA_FILE, RANDOM_STATE, TEST_SIZE

df = pd.read_csv(PROCESSED_DATA_FILE)
X = df['clean_text']; y = df['label_encoded']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
print(f'Train: {len(X_train):,}  Test: {len(X_test):,}')

In [ ]:
# Quick training (no GridSearch for notebook speed)
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
from src.config import MAX_FEATURES, NGRAM_RANGE

models = {
    'Naive Bayes':         MultinomialNB(alpha=0.1),
    'Logistic Regression': LogisticRegression(C=1, max_iter=500, class_weight='balanced'),
    'Random Forest':       RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
}

results = {}
for name, clf in models.items():
    pipe = Pipeline([('tfidf', TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=NGRAM_RANGE, sublinear_tf=True)), ('clf', clf)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    f1 = f1_score(y_test, y_pred)
    results[name] = {'f1': f1, 'pipe': pipe, 'y_pred': y_pred}
    print(f'{name:<25} F1={f1:.4f}')

In [ ]:
# Confusion matrices side by side
from sklearn.metrics import confusion_matrix
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Ham','Spam'], yticklabels=['Ham','Spam'])
    ax.set_title(f'{name}  F1={r["f1"]:.3f}')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.show()

In [ ]:
# Bar chart comparison
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score

metric_data = {}
for name, r in results.items():
    yp = r['y_pred']
    metric_data[name] = {
        'Accuracy':  accuracy_score(y_test, yp),
        'Precision': precision_score(y_test, yp),
        'Recall':    recall_score(y_test, yp),
        'F1':        f1_score(y_test, yp),
    }

df_metrics = pd.DataFrame(metric_data).T
df_metrics.plot(kind='bar', figsize=(10,5), ylim=(0.9,1.01), edgecolor='white')
plt.title('Model Comparison'); plt.ylabel('Score'); plt.xticks(rotation=15)
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()
print(df_metrics.round(4))